# 🔬 Frontier Models Benchmark Suite v2 (10-Model Cohort • 6-Panel Diagnostic Engine)
### NVIDIA A100 Colab Runner (`eval_other.ipynb`)

Notebook này thực thi bộ **6-Panel Frontier Diagnostic Suite (v2)** trên **10 mô hình mã nguồn mở (0.5B đến 16B tham số, 100% Ungated)** xen kẽ giữa **Dense** và **Sparse Mixture-of-Experts (MoE)** trên GPU NVIDIA A100, phục vụ trực tiếp cho thiết kế kiến trúc **Alba (GOR-v2 + OS-MoE + GEO-Muon)**.

### Hệ Thống 6 Nhóm Chỉ Số Học Thuật Chuẩn Hóa (Đã Khắc Phục Toàn Bộ Sai Số Đo Lường v1):
1. **Panel 1 — GOR-v2 Residual Dynamics ($\mathcal{R}_{\text{conflict}}(l)$ & $\eta_\perp(l)$):** Tỷ lệ xung đột triệt tiêu ($\cos(x_l, u_l) < 0$) và tỷ lệ đổi mới trực giao ($1 - \cos^2(x_l, u_l)$) trên luồng residual thực (đã loại bỏ nhiễu `RMSNorm` cuối).
2. **Panel 2 — GOR-v2 Representation Geometry ($\widetilde{\text{erank}}_{\text{norm}}(X_l)$, $BI(l)$ & $1 - \text{CKA}(X_l, X_{l+1})$):** Effective Rank chuẩn hóa L2 theo từng token (khử nhiễu Massive Activation / Attention Sink), chia cho $\min(T, d)$, kết hợp Block Influence và Linear CKA Renewal.
3. **Panel 3 — GOR-v2 Layer Indispensability ($\Delta \mathcal{L}_{\text{rel}}(l)$ trên trục Log):** Độ nhạy cảm giảm tầng tương đối $(\mathcal{L}_{\text{drop}(l)} - \mathcal{L}_{\text{base}}) / \mathcal{L}_{\text{base}} \times 100\%$ hiển thị trên thang Logarit, luôn đo tầng cuối cùng `L-1` và hỗ trợ đa kiến trúc (`Llama`, `Qwen2`, `GPTNeoX/Pythia`, `OLMoE`, `GraniteMoe`, `DeepSeekV2`).
4. **Panel 4 — OS-MoE Routing Equilibrium & Prototype Orthogonality ($\hat{H}_{\text{route}}(l)$, $\mathcal{D}_{\text{ortho}}(l)$, $\text{MaxVio}$, $\text{Max LSE}$):** Entropy định tuyến chuẩn hóa theo $\ln N$, Mutual Information Token-Expert, và độ lệch trực giao $\mathcal{D}_{\text{ortho}} = \|\hat{W}_{\text{gate}}\hat{W}_{\text{gate}}^T - I\|_F / \sqrt{N(N-1)}$ của ma trận vector mẫu router.
5. **Panel 5 — GEO-Muon Spectral Isometry ($\widetilde{\text{srank}}(W_l)$):** Tỷ lệ hạng ổn định phổ (Stable Rank Ratio) $\|W\|_F^2 / (\|W\|_2^2 \min(m,n))$ của các ma trận trọng số Attention & FFN qua từng độ sâu.
6. **Panel 6 — Tokenizer-Agnostic Pareto Frontier ($\text{BPB}$ & $\text{Val PPL}$ vs. Active Params):** Đánh giá đồng thời **Bits-Per-Byte (BPB)** (bất biến với bộ từ vựng Tokenizer) và **Validation Perplexity (PPL)** theo số tham số kích hoạt trên mỗi token.


In [ ]:
import os
import gc
import math
import json
import base64
from io import BytesIO
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import PreTrainedModel, cache_utils
from transformers.utils import import_utils

if not hasattr(import_utils, "is_torch_fx_available"):
    import_utils.is_torch_fx_available = lambda: False

if hasattr(PreTrainedModel, "_initialize_weights") and not getattr(PreTrainedModel, "_safe_init_patched", False):
    _orig_init_weights = PreTrainedModel._initialize_weights
    def _safe_initialize_weights(self, module, is_remote_code=False):
        if getattr(module, "_is_hf_initialized", False):
            return
        params = list(module.parameters(recurse=False))
        buffers = [b for b in module.buffers(recurse=False) if b is not None]
        if (params or buffers) and all(getattr(p, "_is_hf_initialized", False) for p in params) and all(getattr(b, "_is_hf_initialized", False) for b in buffers):
            module._is_hf_initialized = True
            return
        return _orig_init_weights(self, module, True)
    PreTrainedModel._initialize_weights = _safe_initialize_weights
    PreTrainedModel._safe_init_patched = True

if hasattr(cache_utils, "DynamicCache"):
    if not hasattr(cache_utils.DynamicCache, "from_legacy_cache"):
        cache_utils.DynamicCache.from_legacy_cache = staticmethod(
            lambda past_key_values=None: past_key_values if isinstance(past_key_values, cache_utils.DynamicCache) else cache_utils.DynamicCache()
        )
    if not hasattr(cache_utils.DynamicCache, "get_usable_length"):
        cache_utils.DynamicCache.get_usable_length = lambda self, new_seq_length, layer_idx=0: self.get_seq_length(layer_idx)
    if not hasattr(cache_utils.DynamicCache, "get_max_length"):
        cache_utils.DynamicCache.get_max_length = lambda self: None

os.system("nvidia-smi")
os.system("pip install -q transformers accelerate datasets matplotlib scipy tiktoken blobfile")

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("GPU Memory Total (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))


## 1. Danh Mục 10 Mô Hình Đối Chứng Ungated (0.5B – 16B: Iso-Tokenizer Scaling, Cross-Family Dense & Sparse MoE)
Danh mục 10 mô hình mặc định hoàn toàn **Ungated** (chạy trực tiếp trên Colab không cần `HF_TOKEN`), chia thành 3 nhóm đối chứng khoa học:
- **Nhóm 1 — Đường cong Scaling cùng Tokenizer (`Qwen2.5` Family):** `Qwen-2.5-0.5B`, `Qwen-2.5-1.5B`, `Qwen-2.5-3B` (Backbone mục tiêu của Alba), `Qwen-2.5-7B`.
- **Nhóm 2 — Đối chứng Dense Đa Họ (~2B–3B):** `SmolLM2-1.7B` (HuggingFace 11T tokens), `Pythia-2.8B` (EleutherAI Parallel Residual GPTNeoX).
- **Nhóm 3 — Đối chứng Frontier Sparse MoE (0.8B–3.0B Active):** `Granite-3.1-3B-A800M` (IBM 40 experts, top-8), `OLMoE-1B-7B` (AllenAI 64 experts, top-8), `Qwen1.5-MoE-A2.7B` (Alibaba Shared+Routed experts, top-4), `Moonlight-16B-A3B` (Moonshot Muon-trained MoE, top-6).


In [ ]:
FRONTIER_MODELS = {
    "Qwen-2.5-0.5B": {
        "repo_id": "Qwen/Qwen2.5-0.5B",
        "type": "Dense",
        "family": "Qwen2.5",
        "total_params_b": 0.49,
        "active_params_b": 0.49,
        "default_layers": 24,
        "d_model": 896,
        "description": "Compact Qwen2.5 scaling anchor (0.49B dense)"
    },
    "Qwen-2.5-1.5B": {
        "repo_id": "Qwen/Qwen2.5-1.5B",
        "type": "Dense",
        "family": "Qwen2.5",
        "total_params_b": 1.54,
        "active_params_b": 1.54,
        "default_layers": 28,
        "d_model": 1536,
        "description": "Mid-small Qwen2.5 scaling anchor (1.54B dense)"
    },
    "Qwen-2.5-3B": {
        "repo_id": "Qwen/Qwen2.5-3B",
        "type": "Dense",
        "family": "Qwen2.5",
        "total_params_b": 3.09,
        "active_params_b": 3.09,
        "default_layers": 36,
        "d_model": 2048,
        "description": "Primary 3B dense target backbone (36 layers, deep-narrow)"
    },
    "Qwen-2.5-7B": {
        "repo_id": "Qwen/Qwen2.5-7B",
        "type": "Dense",
        "family": "Qwen2.5",
        "total_params_b": 7.61,
        "active_params_b": 7.61,
        "default_layers": 28,
        "d_model": 3584,
        "description": "Upper Qwen2.5 dense scaling bound (7.61B dense)"
    },
    "SmolLM2-1.7B": {
        "repo_id": "HuggingFaceTB/SmolLM2-1.7B",
        "type": "Dense",
        "family": "SmolLM2",
        "total_params_b": 1.71,
        "active_params_b": 1.71,
        "default_layers": 24,
        "d_model": 2048,
        "description": "HuggingFace over-trained compact dense model (11T tokens)"
    },
    "Pythia-2.8B": {
        "repo_id": "EleutherAI/pythia-2.8b",
        "type": "Dense",
        "family": "Pythia",
        "total_params_b": 2.78,
        "active_params_b": 2.78,
        "default_layers": 32,
        "d_model": 2560,
        "description": "EleutherAI GPTNeoX parallel-residual 2.8B dense baseline"
    },
    "Granite-3.1-3B-A800M": {
        "repo_id": "ibm-granite/granite-3.1-3b-a800m-base",
        "type": "MoE",
        "family": "Granite-MoE",
        "total_params_b": 3.30,
        "active_params_b": 0.80,
        "default_layers": 32,
        "d_model": 1536,
        "top_k": 8,
        "num_experts": 40,
        "description": "IBM sub-1B active fine-grained 40-expert MoE (top-8)"
    },
    "OLMoE-1B-7B": {
        "repo_id": "allenai/OLMoE-1B-7B-0924",
        "type": "MoE",
        "family": "OLMoE",
        "total_params_b": 6.92,
        "active_params_b": 1.30,
        "default_layers": 16,
        "d_model": 2048,
        "top_k": 8,
        "num_experts": 64,
        "description": "AllenAI open 64-expert MoE (top-8 active, 1.3B active)"
    },
    "Qwen1.5-MoE-A2.7B": {
        "repo_id": "Qwen/Qwen1.5-MoE-A2.7B",
        "type": "MoE",
        "family": "Qwen-MoE",
        "total_params_b": 14.30,
        "active_params_b": 2.70,
        "default_layers": 24,
        "d_model": 2048,
        "top_k": 4,
        "num_experts": 60,
        "description": "Alibaba hybrid MoE with 4 shared + 60 routed experts (top-4)"
    },
    "Moonlight-16B-A3B": {
        "repo_id": "moonshotai/Moonlight-16B-A3B",
        "type": "MoE",
        "family": "Moonlight",
        "total_params_b": 15.70,
        "active_params_b": 2.97,
        "default_layers": 27,
        "d_model": 2048,
        "top_k": 6,
        "num_experts": 64,
        "description": "Moonshot 16B MoE trained with spectral Muon optimizer",
        "trust_remote_code": True
    },
    "DeepSeek-V2-Lite": {
        "repo_id": "deepseek-ai/DeepSeek-V2-Lite",
        "type": "MoE",
        "family": "DeepSeek-MoE",
        "total_params_b": 15.70,
        "active_params_b": 2.40,
        "default_layers": 27,
        "d_model": 2048,
        "top_k": 6,
        "num_experts": 64,
        "description": "DeepSeek fine-grained MoE with Multi-head Latent Attention",
        "trust_remote_code": True
    },
    "Qwen-2.5-14B": {
        "repo_id": "Qwen/Qwen2.5-14B",
        "type": "Dense",
        "family": "Qwen2.5",
        "total_params_b": 14.70,
        "active_params_b": 14.70,
        "default_layers": 48,
        "d_model": 5120,
        "description": "Heavyweight 14.7B dense reference model"
    },
    "Llama-3.2-3B": {
        "repo_id": "meta-llama/Llama-3.2-3B",
        "type": "Dense",
        "family": "Llama-3",
        "total_params_b": 3.21,
        "active_params_b": 3.21,
        "default_layers": 28,
        "d_model": 3072,
        "description": "Meta compact 3B dense model (Requires HF_TOKEN)"
    },
    "Gemma-2-2B": {
        "repo_id": "google/gemma-2-2b",
        "type": "Dense",
        "family": "Gemma-2",
        "total_params_b": 2.61,
        "active_params_b": 2.61,
        "default_layers": 26,
        "d_model": 2304,
        "description": "Google DeepMind 2.6B logit soft-capping baseline (Requires HF_TOKEN)"
    },
    "Llama-3.1-8B": {
        "repo_id": "meta-llama/Llama-3.1-8B",
        "type": "Dense",
        "family": "Llama-3",
        "total_params_b": 8.03,
        "active_params_b": 8.03,
        "default_layers": 32,
        "d_model": 4096,
        "description": "Meta 8B dense baseline (Requires HF_TOKEN)"
    },
    "Gemma-2-9B": {
        "repo_id": "google/gemma-2-9b",
        "type": "Dense",
        "family": "Gemma-2",
        "total_params_b": 9.24,
        "active_params_b": 9.24,
        "default_layers": 42,
        "d_model": 3584,
        "description": "Google DeepMind 9B dense baseline (Requires HF_TOKEN)"
    }
}

CALIBRATION_CORPUS = [
    "Large language models have fundamentally transformed artificial intelligence through autoregressive self-supervised pre-training across massive web-scale corpora.",
    "In mathematical logic and deduction, consider a sequence of formal derivations where each proposition follows necessarily from established axioms and inference rules.",
    "def quicksort(arr):\n    if len(arr) <= 1:\n        return arr\n    pivot = arr[len(arr) // 2]\n    left = [x for x in arr if x < pivot]\n    middle = [x for x in arr if x == pivot]\n    right = [x for x in arr if x > pivot]\n    return quicksort(left) + middle + quicksort(right)",
    "The spectral theorem asserts that any symmetric or Hermitian matrix can be decomposed into an orthogonal basis of eigenvectors with real corresponding eigenvalues.",
    "Mixture-of-Experts architectures replace monolithic dense feed-forward networks with sparse conditional routing, distributing tokens dynamically across specialized sub-networks.",
    "Effective depth in deep residual networks measures the actual representation renewal rate, revealing that standard additive connections experience representation collapse in later layers.",
    "A quantum algorithm demonstrates speedup by exploiting coherent superposition and constructive interference over an exponentially large Hilbert state space.",
    "Natural language semantics exhibits compositionality, where the syntactic structure determines the recursive combination of lexical meanings into higher-order assertions."
]

print("Loaded Model Registry with", len(FRONTIER_MODELS), "frontier models.")


## 2. Universal 6-Metric Diagnostic Engine (v2)
Tích hợp đầy đủ các hiệu chỉnh toán học:
- **Pre-Final-Norm Residual Hook:** Thu nhận trực tiếp đầu ra tầng `layers[-1]` trước khi đi qua `final_norm` để loại bỏ bước nhảy giả tạo ở tầng cuối.
- **GOR-v2 Mechanism:** Đo tỷ lệ xung đột $\mathcal{R}_{\text{conflict}}(l)$ và tỷ lệ đổi mới trực giao $\eta_\perp(l)$ giữa trạng thái $x_l$ và cập nhật $u_l = x_{l+1} - x_l$.
- **Token-L2-Normalized Effective Rank & Linear CKA:** Chuẩn hóa độ dài vector của từng token trước khi dịch tâm nhằm triệt tiêu hiện tượng Attention Sink chi phối giá trị kỳ dị $\sigma_1$, chuẩn hóa theo $\min(T, d)$.
- **Log-Scale Relative Layer-Drop:** Hỗ trợ `Llama`, `Qwen2`, `GPTNeoX/Pythia`, luôn đánh giá tầng cuối cùng `L-1`, biểu diễn phần trăm tăng Loss tương đối trên thang Log.
- **OS-MoE Routing & Prototype Orthogonality:** Sử dụng `top_k` chính xác của từng mô hình, đo $\hat{H}_{\text{route}}$, Mutual Information $I(X;E)$, và $\mathcal{D}_{\text{ortho}}(W_{\text{gate}})$.
- **GEO-Muon Spectral Stable Rank & BPB:** Đo $\widetilde{\text{srank}}(W_l)$ bằng Power Iteration trên GPU và chuẩn hóa Bits-Per-Byte (BPB) độc lập Tokenizer.


In [ ]:
import math
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, PreTrainedModel, cache_utils
from transformers.utils import import_utils
import traceback

if not hasattr(import_utils, "is_torch_fx_available"):
    import_utils.is_torch_fx_available = lambda: False

if hasattr(PreTrainedModel, "_initialize_weights") and not getattr(PreTrainedModel, "_safe_init_patched", False):
    _orig_init_weights = PreTrainedModel._initialize_weights
    def _safe_initialize_weights(self, module, is_remote_code=False):
        if getattr(module, "_is_hf_initialized", False):
            return
        params = list(module.parameters(recurse=False))
        buffers = [b for b in module.buffers(recurse=False) if b is not None]
        if (params or buffers) and all(getattr(p, "_is_hf_initialized", False) for p in params) and all(getattr(b, "_is_hf_initialized", False) for b in buffers):
            module._is_hf_initialized = True
            return
        return _orig_init_weights(self, module, True)
    PreTrainedModel._initialize_weights = _safe_initialize_weights
    PreTrainedModel._safe_init_patched = True

if hasattr(cache_utils, "DynamicCache"):
    if not hasattr(cache_utils.DynamicCache, "from_legacy_cache"):
        cache_utils.DynamicCache.from_legacy_cache = staticmethod(
            lambda past_key_values=None: past_key_values if isinstance(past_key_values, cache_utils.DynamicCache) else cache_utils.DynamicCache()
        )
    if not hasattr(cache_utils.DynamicCache, "get_usable_length"):
        cache_utils.DynamicCache.get_usable_length = lambda self, new_seq_length, layer_idx=0: self.get_seq_length(layer_idx)
    if not hasattr(cache_utils.DynamicCache, "get_max_length"):
        cache_utils.DynamicCache.get_max_length = lambda self: None

def get_decoder_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "gpt_neox") and hasattr(model.gpt_neox, "layers"):
        return model.gpt_neox.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "layers"):
        return model.layers
    return None

def compute_gor_conflict_and_innovation(hidden_states):
    conflict_rates = []
    ortho_innovations = []
    for i in range(len(hidden_states) - 1):
        x_l = hidden_states[i].squeeze(0).float()
        x_next = hidden_states[i + 1].squeeze(0).float()
        u_l = x_next - x_l
        dot = (x_l * u_l).sum(dim=-1)
        norm_x = torch.norm(x_l, p=2, dim=-1)
        norm_u = torch.norm(u_l, p=2, dim=-1)
        rho = (dot / (norm_x * norm_u + 1e-8)).clamp(-1.0, 1.0)
        conflict_pct = float((rho < 0.0).float().mean().item() * 100.0)
        ortho_pct = float((1.0 - rho * rho).mean().item() * 100.0)
        conflict_rates.append(round(conflict_pct, 2))
        ortho_innovations.append(round(ortho_pct, 2))
    return conflict_rates, ortho_innovations

def compute_erank_metric(hidden_states):
    erank_norm_ratios = []
    raw_eranks = []
    for hs in hidden_states:
        flat = hs.squeeze(0).float()
        normed = F.normalize(flat, p=2, dim=-1)
        centered = normed - normed.mean(dim=0, keepdim=True)
        s = torch.linalg.svdvals(centered)
        s_sum = s.sum() + 1e-12
        p = s / s_sum
        p_nz = p[p > 1e-12]
        entropy = -(p_nz * torch.log(p_nz)).sum()
        erank_val = float(torch.exp(entropy).item())
        max_possible_rank = max(min(flat.size(0), flat.size(1)), 1)
        raw_eranks.append(round(erank_val, 2))
        erank_norm_ratios.append(round(erank_val / max_possible_rank, 4))
    return erank_norm_ratios, raw_eranks

def compute_block_influence_and_cka(hidden_states):
    bi_list = []
    cka_renewal_list = []
    for i in range(len(hidden_states) - 1):
        h_curr = hidden_states[i].squeeze(0).float()
        h_next = hidden_states[i + 1].squeeze(0).float()
        norm_curr = torch.norm(h_curr, p=2, dim=-1)
        norm_next = torch.norm(h_next, p=2, dim=-1)
        dot = (h_curr * h_next).sum(dim=-1)
        cos = dot / (norm_curr * norm_next + 1e-12)
        bi_list.append(round(float((1.0 - cos.mean()).item()), 4))

        x_n = F.normalize(h_curr, p=2, dim=-1)
        y_n = F.normalize(h_next, p=2, dim=-1)
        x_c = x_n - x_n.mean(dim=0, keepdim=True)
        y_c = y_n - y_n.mean(dim=0, keepdim=True)
        gram_x = x_c @ x_c.T
        gram_y = y_c @ y_c.T
        hsic = (gram_x * gram_y).sum()
        denom = torch.linalg.matrix_norm(gram_x, ord="fro") * torch.linalg.matrix_norm(gram_y, ord="fro") + 1e-12
        cka_val = float((hsic / denom).clamp(0.0, 1.0).item())
        cka_renewal_list.append(round(1.0 - cka_val, 4))
    return bi_list, cka_renewal_list

def _eval_causal_loss(model, input_ids):
    outputs = model(input_ids, labels=input_ids, use_cache=False)
    if hasattr(outputs, "loss") and outputs.loss is not None and not torch.isnan(outputs.loss):
        return float(outputs.loss.item())
    logits = outputs.logits
    shift_logits = logits[..., :-1, :].contiguous().float()
    shift_labels = input_ids[..., 1:].contiguous()
    loss_fct = torch.nn.CrossEntropyLoss()
    return float(loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1)).item())

def compute_layer_drop_sensitivity(model, input_ids):
    with torch.no_grad():
        base_loss = _eval_causal_loss(model, input_ids)

    layers = get_decoder_layers(model)
    raw_delta_list = []
    rel_delta_pct_list = []
    depth_pct_axis = []

    if layers is not None and len(layers) > 0:
        total_layers = len(layers)
        if total_layers <= 36:
            eval_indices = list(range(total_layers))
        else:
            eval_indices = sorted(list(set(list(range(0, total_layers, 2)) + [total_layers - 1])))

        for idx in eval_indices:
            target_layer = layers[idx]

            def bypass_hook(module, args, output):
                inp = args[0] if len(args) > 0 else output
                if isinstance(output, tuple):
                    return (inp,) + output[1:]
                return inp

            hook = target_layer.register_forward_hook(bypass_hook)
            try:
                with torch.no_grad():
                    dropped_loss = _eval_causal_loss(model, input_ids)
                raw_delta = max(0.0, dropped_loss - base_loss)
                rel_pct = max(0.01, ((dropped_loss - base_loss) / max(base_loss, 1e-8)) * 100.0)
            except Exception:
                raw_delta = 0.0
                rel_pct = 0.01
            finally:
                hook.remove()

            raw_delta_list.append(round(raw_delta, 4))
            rel_delta_pct_list.append(round(rel_pct, 3))
            depth_pct_axis.append(round((idx / max(total_layers - 1, 1)) * 100.0, 2))

    return raw_delta_list, rel_delta_pct_list, depth_pct_axis, base_loss

def compute_router_prototype_orthogonality(model, d_model):
    defects = []
    seen_ptrs = set()
    for name, module in model.named_modules():
        cls_name = type(module).__name__.lower()
        is_router_cand = (
            name.endswith(".gate")
            or name.endswith(".router")
            or name.endswith(".router.layer")
            or "router" in cls_name
            or cls_name.endswith("gate")
            or "topkrouter" in cls_name
            or "topkgating" in cls_name
        ) and not name.endswith("gate_proj") and not name.endswith("shared_expert_gate")
        if not is_router_cand:
            continue
        w = getattr(module, "weight", None)
        if w is None and hasattr(module, "layer"):
            w = getattr(module.layer, "weight", None)
        if w is None or not torch.is_tensor(w) or w.dim() != 2:
            continue
        ptr = w.data_ptr()
        if ptr in seen_ptrs:
            continue
        n_exp, feat_dim = w.size(0), w.size(1)
        if n_exp < 4 or n_exp > 512 or (d_model and feat_dim != d_model):
            continue
        seen_ptrs.add(ptr)
        with torch.no_grad():
            w_norm = F.normalize(w.detach().float(), p=2, dim=-1)
            gram = w_norm @ w_norm.T
            eye = torch.eye(n_exp, device=gram.device, dtype=gram.dtype)
            defect = float(torch.linalg.matrix_norm(gram - eye, ord="fro").item() / math.sqrt(n_exp * (n_exp - 1)))
            defects.append(round(defect, 4))
    return defects

def _CompileMoEStats(counts_list, norm_entropies, mi_list, max_lse_val, ortho_defects):
    vios = []
    for counts in counts_list:
        expected = counts.sum().item() / max(counts.numel(), 1)
        vios.append((counts.max().item() - expected) / (expected + 1e-12))
    is_moe = len(vios) > 0 or len(ortho_defects) > 0
    return {
        "is_moe": is_moe,
        "max_vio_global": round(float(np.mean(vios)), 3) if vios else 0.0,
        "max_vio_batch": round(float(np.max(vios)), 3) if vios else 0.0,
        "max_lse": round(float(max_lse_val), 2),
        "routing_entropy_pct": round(float(np.mean(norm_entropies)), 2) if norm_entropies else 0.0,
        "routing_mi_pct": round(float(np.mean(mi_list)), 2) if mi_list else 0.0,
        "router_ortho_defect": round(float(np.mean(ortho_defects)), 4) if ortho_defects else 0.0,
        "layer_routing_entropy_curve": [round(float(x), 2) for x in norm_entropies],
        "layer_ortho_defect_curve": ortho_defects
    }

def analyze_moe_router(model, input_ids, model_meta=None):
    cfg = model.config
    d_model = (model_meta or {}).get("d_model") or getattr(cfg, "hidden_size", None)
    ortho_defects = compute_router_prototype_orthogonality(model, d_model)
    empty = _CompileMoEStats([], [], [], 0.0, ortho_defects)

    top_k = (
        (model_meta or {}).get("top_k")
        or getattr(cfg, "num_experts_per_tok", None)
        or getattr(cfg, "top_k", None)
        or getattr(cfg, "moe_top_k", None)
        or 2
    )
    n_experts_cfg = (
        (model_meta or {}).get("num_experts")
        or getattr(cfg, "n_routed_experts", None)
        or getattr(cfg, "num_experts", None)
        or getattr(cfg, "num_local_experts", None)
    )

    try:
        with torch.no_grad():
            outputs = model(input_ids, output_router_logits=True, use_cache=False)
        router_logits = getattr(outputs, "router_logits", None)
        if router_logits is not None:
            counts_list, norm_entropies, mi_list, max_lse_val = [], [], [], 0.0
            for r in router_logits:
                if r is None or not torch.is_tensor(r):
                    continue
                flat = r.reshape(-1, r.size(-1)).float()
                n_exp = flat.size(-1)
                if n_exp < 2:
                    continue
                log_n = math.log(n_exp)
                max_lse_val = max(max_lse_val, torch.logsumexp(flat, dim=-1).max().item())
                probs = torch.softmax(flat, dim=-1)
                k_eff = min(int(top_k), n_exp)
                idx = torch.topk(probs, k=k_eff, dim=-1).indices
                counts_list.append(torch.bincount(idx.reshape(-1), minlength=n_exp).float())
                mp = probs.mean(dim=0)
                mp_nz = mp[mp > 1e-12]
                h_marg = -(mp_nz * torch.log(mp_nz)).sum().item()
                h_cond = -(probs * torch.log(probs.clamp(min=1e-12))).sum(dim=-1).mean().item()
                norm_entropies.append((h_marg / log_n) * 100.0)
                mi_list.append(max(0.0, (h_marg - h_cond) / log_n) * 100.0)
            stats = _CompileMoEStats(counts_list, norm_entropies, mi_list, max_lse_val, ortho_defects)
            if len(counts_list) > 0:
                return stats
    except Exception:
        pass

    try:
        captured = []
        hooks = []
        for name, module in model.named_modules():
            cls_name = type(module).__name__.lower()
            if (
                name.endswith(".gate")
                or name.endswith(".router")
                or "router" in cls_name
                or cls_name.endswith("gate")
                or "topkrouter" in cls_name
                or "topkgating" in cls_name
            ) and not name.endswith("gate_proj") and not name.endswith("shared_expert_gate"):
                def _make_router_hook(mod_ref):
                    def _hook_fn(m, a, o):
                        captured.append((mod_ref, a, o))
                    return _hook_fn
                hooks.append(module.register_forward_hook(_make_router_hook(module)))
        if hooks:
            with torch.no_grad():
                model(input_ids, use_cache=False)
            for h in hooks:
                h.remove()
        counts_list, norm_entropies, mi_list, max_lse_val = [], [], [], 0.0
        for mod_ref, a, o in captured:
            tensors = o if isinstance(o, (tuple, list)) else [o]
            logits, indices = None, None
            for t in tensors:
                if not torch.is_tensor(t):
                    continue
                if t.is_floating_point() and t.dim() >= 2:
                    last_d = t.size(-1)
                    if n_experts_cfg is not None:
                        if last_d == n_experts_cfg:
                            logits = t
                    elif 4 <= last_d <= 256 and last_d != int(top_k):
                        logits = t
                elif (not t.is_floating_point()) and t.dim() >= 1:
                    if indices is None:
                        indices = t
            if logits is None and len(a) > 0 and torch.is_tensor(a[0]):
                w_gate = getattr(mod_ref, "weight", None)
                if w_gate is None and hasattr(mod_ref, "layer"):
                    w_gate = getattr(mod_ref.layer, "weight", None)
                if w_gate is not None and torch.is_tensor(w_gate) and w_gate.dim() == 2:
                    h_in = a[0].reshape(-1, a[0].size(-1)).float()
                    if h_in.size(-1) == w_gate.size(1):
                        logits = F.linear(h_in, w_gate.detach().float())
            if logits is not None:
                flat = logits.reshape(-1, logits.size(-1)).float()
                n_exp = flat.size(-1)
                log_n = math.log(max(n_exp, 2))
                max_lse_val = max(max_lse_val, torch.logsumexp(flat, dim=-1).max().item())
                probs = torch.softmax(flat, dim=-1)
                k_eff = min(int(top_k), n_exp)
                idx = indices if (indices is not None and indices.numel() > 0) else torch.topk(probs, k=k_eff, dim=-1).indices
                counts_list.append(torch.bincount(idx.reshape(-1).clamp(min=0, max=n_exp - 1), minlength=n_exp).float())
                mp = probs.mean(dim=0)
                mp_nz = mp[mp > 1e-12]
                h_marg = -(mp_nz * torch.log(mp_nz)).sum().item()
                h_cond = -(probs * torch.log(probs.clamp(min=1e-12))).sum(dim=-1).mean().item()
                norm_entropies.append((h_marg / log_n) * 100.0)
                mi_list.append(max(0.0, (h_marg - h_cond) / log_n) * 100.0)
            elif indices is not None and n_experts_cfg:
                c = torch.bincount(indices.reshape(-1).clamp(min=0, max=n_experts_cfg - 1), minlength=n_experts_cfg).float()
                counts_list.append(c)
                p = c / (c.sum() + 1e-12)
                p_nz = p[p > 1e-12]
                h_marg = -(p_nz * torch.log(p_nz)).sum().item()
                norm_entropies.append((h_marg / math.log(max(n_experts_cfg, 2))) * 100.0)
        stats = _CompileMoEStats(counts_list, norm_entropies, mi_list, max_lse_val, ortho_defects)
        if stats["is_moe"]:
            return stats
    except Exception:
        pass

    return empty

def _matrix_srank_ratio(w_2d):
    w = w_2d.detach().float()
    m, n = w.size(0), w.size(1)
    min_dim = min(m, n)
    if min_dim < 2:
        return None
    frob_sq = float((w * w).sum().item())
    v = torch.linspace(-1.0, 1.0, n, device=w.device, dtype=torch.float32)
    v = v / (torch.norm(v, p=2) + 1e-12)
    for _ in range(10):
        u = w @ v
        u_norm = torch.norm(u, p=2)
        if u_norm < 1e-12:
            break
        u = u / u_norm
        v = w.T @ u
        v_norm = torch.norm(v, p=2)
        if v_norm < 1e-12:
            break
        v = v / v_norm
    sigma_max = float(torch.norm(w @ v, p=2).item())
    srank = frob_sq / (sigma_max * sigma_max + 1e-12)
    return min(1.0, max(0.0, srank / min_dim))

def compute_spectral_stable_rank(model):
    layers = get_decoder_layers(model)
    if layers is None:
        return []
    layer_srank_curve = []
    with torch.no_grad():
        for layer in layers:
            ratios = []
            for name, param in layer.named_parameters():
                if param is None or not param.is_floating_point():
                    continue
                if param.dim() == 2 and min(param.size(0), param.size(1)) >= 64:
                    r = _matrix_srank_ratio(param)
                    if r is not None:
                        ratios.append(r)
                elif param.dim() == 3 and min(param.size(1), param.size(2)) >= 64:
                    r = _matrix_srank_ratio(param[0])
                    if r is not None:
                        ratios.append(r)
                if len(ratios) >= 8:
                    break
            if ratios:
                layer_srank_curve.append(round(float(np.mean(ratios)), 4))
    return layer_srank_curve

print("Universal 6-Metric Diagnostic Engine v2 successfully compiled.")


In [ ]:
def run_model_benchmark(model_key, model_meta, token_text_list, hf_token=None):
    repo_id = model_meta["repo_id"]
    print("\n=======================================================")
    print(f"🚀 Evaluating: {model_key} [{model_meta['type']}] ({model_meta['total_params_b']}B Total / {model_meta['active_params_b']}B Active)")
    print("=======================================================")

    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(
        repo_id,
        token=hf_token,
        trust_remote_code=model_meta.get("trust_remote_code", False)
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    combined_text = "\n\n".join(token_text_list)
    encoded = tokenizer(
        combined_text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )
    input_ids = encoded["input_ids"].to(device)
    decoded_eval_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    num_utf8_bytes = max(len(decoded_eval_text.encode("utf-8")), 1)
    num_target_tokens = max(input_ids.size(1) - 1, 1)

    trust_rc = model_meta.get("trust_remote_code", False)
    cfg = AutoConfig.from_pretrained(repo_id, token=hf_token, trust_remote_code=trust_rc)
    if trust_rc and getattr(cfg, "rope_scaling", None) is not None and isinstance(cfg.rope_scaling, dict):
        if "type" not in cfg.rope_scaling:
            rt = cfg.rope_scaling.get("rope_type", "default")
            if rt == "default":
                cfg.rope_scaling = None
            else:
                cfg.rope_scaling["type"] = rt

    model = AutoModelForCausalLM.from_pretrained(
        repo_id,
        config=cfg,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        token=hf_token,
        trust_remote_code=trust_rc
    )
    for mod in model.modules():
        if hasattr(mod, "_set_cos_sin_cache") and hasattr(mod, "dim") and hasattr(mod, "base"):
            rot_dev = mod.inv_freq.device if hasattr(mod, "inv_freq") and mod.inv_freq.device.type != "meta" else torch.device(device)
            inv_freq = 1.0 / (float(mod.base) ** (torch.arange(0, int(mod.dim), 2, dtype=torch.float32, device=rot_dev) / float(mod.dim)))
            mod.register_buffer("inv_freq", inv_freq, persistent=False)
            max_pos = getattr(mod, "max_position_embeddings", 8192) or 8192
            mod._set_cos_sin_cache(seq_len=max_pos, device=rot_dev, dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)
    model.eval()

    warmup_len = min(64, input_ids.size(1))
    with torch.no_grad():
        _ = model(input_ids[:, :warmup_len], use_cache=False)
    if device == "cuda":
        torch.cuda.synchronize()

    layers = get_decoder_layers(model)
    raw_last_hidden = []
    last_hook = None
    if layers is not None and len(layers) > 0:
        def _capture_pre_norm(module, args, output):
            h = output[0] if isinstance(output, (tuple, list)) else output
            if torch.is_tensor(h):
                raw_last_hidden.append(h.detach())
        last_hook = layers[-1].register_forward_hook(_capture_pre_norm)

    start_time = torch.cuda.Event(enable_timing=True) if device == "cuda" else None
    end_time = torch.cuda.Event(enable_timing=True) if device == "cuda" else None

    if start_time:
        start_time.record()

    with torch.no_grad():
        outputs = model(input_ids, output_hidden_states=True, use_cache=False)
        hidden_states = list(outputs.hidden_states)

    if last_hook is not None:
        last_hook.remove()
    if raw_last_hidden and len(hidden_states) >= 2 and raw_last_hidden[-1].shape == hidden_states[-1].shape:
        hidden_states[-1] = raw_last_hidden[-1]

    if end_time:
        end_time.record()
        torch.cuda.synchronize()
        elapsed_ms = max(start_time.elapsed_time(end_time), 1e-3)
    else:
        elapsed_ms = 100.0

    throughput_tok_s = round((input_ids.size(1) / (elapsed_ms / 1000.0)), 1)

    conflict_curve, ortho_innov_curve = compute_gor_conflict_and_innovation(hidden_states)
    erank_norm_curve, raw_erank_curve = compute_erank_metric(hidden_states)
    bi_scores, cka_renewal_curve = compute_block_influence_and_cka(hidden_states)
    layer_drops_raw, layer_drops_rel_pct, drop_depth_axis, base_loss = compute_layer_drop_sensitivity(model, input_ids)
    moe_stats = analyze_moe_router(model, input_ids, model_meta=model_meta)
    srank_curve = compute_spectral_stable_rank(model)

    del model
    del tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    ppl = round(float(np.exp(min(base_loss, 20.0))), 2)
    val_bpb = round(float((num_target_tokens * base_loss) / (num_utf8_bytes * math.log(2.0))), 4)
    half_idx = max(len(layer_drops_rel_pct) // 2, 0)
    deep_drop_pct = round(float(np.mean(layer_drops_rel_pct[half_idx:])), 3) if layer_drops_rel_pct else 0.0
    eff_depth_ratio = round(float(np.mean([1.0 if c > 0.02 else 0.0 for c in cka_renewal_curve])), 3) if cka_renewal_curve else 0.0

    summary = {
        "model_key": model_key,
        "type": model_meta["type"],
        "family": model_meta.get("family", model_meta["type"]),
        "total_params_b": model_meta["total_params_b"],
        "active_params_b": model_meta["active_params_b"],
        "d_model": model_meta["d_model"],
        "num_layers": len(hidden_states) - 1,
        "num_tokens": int(input_ids.size(1)),
        "num_utf8_bytes": int(num_utf8_bytes),
        "base_loss": round(base_loss, 4),
        "val_ppl": ppl,
        "val_bpb": val_bpb,
        "throughput_tok_s": throughput_tok_s,
        "avg_conflict_rate_pct": round(float(np.mean(conflict_curve)), 2) if conflict_curve else 0.0,
        "avg_ortho_innovation_pct": round(float(np.mean(ortho_innov_curve)), 2) if ortho_innov_curve else 0.0,
        "avg_erank_norm_ratio": round(float(np.mean(erank_norm_curve)), 4) if erank_norm_curve else 0.0,
        "final_erank_norm_ratio": erank_norm_curve[-1] if erank_norm_curve else 0.0,
        "avg_bi": round(float(np.mean(bi_scores)), 4) if bi_scores else 0.0,
        "final_layer_bi": round(bi_scores[-1], 4) if bi_scores else 0.0,
        "avg_cka_renewal": round(float(np.mean(cka_renewal_curve)), 4) if cka_renewal_curve else 0.0,
        "effective_depth_ratio": eff_depth_ratio,
        "avg_layer_drop_rel_pct": round(float(np.mean(layer_drops_rel_pct)), 3) if layer_drops_rel_pct else 0.0,
        "deep_layer_drop_rel_pct": deep_drop_pct,
        "final_layer_drop_rel_pct": round(layer_drops_rel_pct[-1], 3) if layer_drops_rel_pct else 0.0,
        "avg_srank_ratio": round(float(np.mean(srank_curve)), 4) if srank_curve else 0.0,
        "is_moe": moe_stats["is_moe"],
        "max_vio_global": moe_stats["max_vio_global"],
        "max_vio_batch": moe_stats["max_vio_batch"],
        "max_lse": moe_stats["max_lse"],
        "routing_entropy_pct": moe_stats["routing_entropy_pct"],
        "routing_mi_pct": moe_stats["routing_mi_pct"],
        "router_ortho_defect": moe_stats["router_ortho_defect"],
        "layer_conflict_curve": conflict_curve,
        "layer_ortho_innovation_curve": ortho_innov_curve,
        "layer_erank_curve": erank_norm_curve,
        "layer_raw_erank_curve": raw_erank_curve,
        "layer_bi_curve": bi_scores,
        "layer_cka_renewal_curve": cka_renewal_curve,
        "layer_drop_curve": layer_drops_raw,
        "layer_drop_rel_pct_curve": layer_drops_rel_pct,
        "layer_drop_depth_axis": drop_depth_axis,
        "layer_srank_curve": srank_curve,
        "layer_routing_entropy_curve": moe_stats["layer_routing_entropy_curve"],
        "layer_ortho_defect_curve": moe_stats["layer_ortho_defect_curve"]
    }

    print(
        f"✅ Finished {model_key}: BPB = {val_bpb} | PPL = {ppl} | "
        f"Conflict = {summary['avg_conflict_rate_pct']}% | erank_norm = {summary['avg_erank_norm_ratio']} | "
        f"srank = {summary['avg_srank_ratio']} | Tok/s = {throughput_tok_s}"
    )
    return summary


## 3. Thực Thi Benchmark 10 Mô Hình Trên GPU A100 (Tự Động Lưu & Phục Hồi Cache v2)
Khối mã dưới đây chạy toàn bộ **10 mô hình đối chứng Ungated** (và tự động bổ sung `Llama-3.2-3B`, `Gemma-2-2B` nếu phát hiện `HF_TOKEN`). Kết quả được lưu tức thời sau mỗi mô hình vào `frontier_benchmark_v2_results.json`.


In [ ]:
import shutil

MODELS_TO_RUN = [
    "Qwen-2.5-0.5B",
    "Qwen-2.5-1.5B",
    "Qwen-2.5-3B",
    "Qwen-2.5-7B",
    "SmolLM2-1.7B",
    "Pythia-2.8B",
    "Granite-3.1-3B-A800M",
    "OLMoE-1B-7B",
    "Qwen1.5-MoE-A2.7B",
    "Moonlight-16B-A3B"
]

try:
    from google.colab import userdata
    HF_AUTH_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_AUTH_TOKEN = os.environ.get("HF_TOKEN", None)

if HF_AUTH_TOKEN:
    for gm in ["Llama-3.2-3B", "Gemma-2-2B"]:
        if gm not in MODELS_TO_RUN:
            MODELS_TO_RUN.append(gm)

CACHE_FILE = "frontier_benchmark_v2_results.json"
benchmark_results = {}
if os.path.exists(CACHE_FILE):
    try:
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            benchmark_results = json.load(f)
        print(f"♻️ Loaded {len(benchmark_results)} cached v2 model results from {CACHE_FILE}")
    except Exception:
        benchmark_results = {}

for model_name in MODELS_TO_RUN:
    if model_name in benchmark_results and "val_bpb" in benchmark_results[model_name] and benchmark_results[model_name].get("val_ppl", 999.0) < 100.0:
        print(f"⏭️ Skipping {model_name} (already evaluated in {CACHE_FILE})")
        continue
    if model_name in FRONTIER_MODELS:
        meta = FRONTIER_MODELS[model_name]
        try:
            res = run_model_benchmark(model_name, meta, CALIBRATION_CORPUS, hf_token=HF_AUTH_TOKEN)
            benchmark_results[model_name] = res
        except Exception as e:
            print(f"❌ Error evaluating {model_name}: {e}")
            traceback.print_exc()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub"), ignore_errors=True)
        with open(CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(benchmark_results, f, indent=2)
        with open("frontier_benchmark_results.json", "w", encoding="utf-8") as f:
            json.dump(benchmark_results, f, indent=2)

print(f"\n🎉 Successfully finished evaluation of {len(benchmark_results)} models. Results saved to {CACHE_FILE}")


## 4. Xuất Báo Cáo HTML Độc Lập 6-Panel (`frontier_models_benchmark_report.html`)
Khối mã dưới đây tự động kết xuất lưới đồ thị **6-Panel (2x3) Publication Grid**, bảng dữ liệu tổng hợp toàn diện 10+ mô hình, nhúng ảnh PNG độ phân giải cao dạng Base64 và tự động tải về máy.


In [ ]:
def generate_html_benchmark_report(results_data, output_html_path="frontier_models_benchmark_report.html"):
    fig, axes = plt.subplots(2, 3, figsize=(22, 13), dpi=150)
    plt.subplots_adjust(hspace=0.35, wspace=0.26)

    ax_gor = axes[0, 0]
    ax_erank = axes[0, 1]
    ax_drop = axes[0, 2]
    ax_moe = axes[1, 0]
    ax_srank = axes[1, 1]
    ax_pareto = axes[1, 2]

    cmap = plt.cm.tab10 if len(results_data) <= 10 else plt.cm.tab20
    colors = cmap(np.linspace(0, 1, max(len(results_data), 1)))

    pareto_points = []

    for (m_key, res), color in zip(results_data.items(), colors):
        label_text = f"{m_key} ({res['type']})"
        ls = "-" if res["type"] == "Dense" else "--"
        marker = "o" if res["type"] == "Dense" else "s"

        conflict_curve = res.get("layer_conflict_curve", [])
        if conflict_curve:
            depth_pct = np.linspace(0, 100, len(conflict_curve))
            ax_gor.plot(depth_pct, conflict_curve, label=label_text, color=color, linestyle=ls, marker=marker, markersize=3.5, alpha=0.85)

        erank_curve = res.get("layer_erank_curve", [])
        if erank_curve:
            depth_pct_e = np.linspace(0, 100, len(erank_curve))
            ax_erank.plot(depth_pct_e, erank_curve, label=label_text, color=color, linestyle=ls, marker=marker, markersize=3.5, alpha=0.85)

        drop_rel = res.get("layer_drop_rel_pct_curve", [])
        drop_axis = res.get("layer_drop_depth_axis", [])
        if drop_rel:
            if not drop_axis or len(drop_axis) != len(drop_rel):
                drop_axis = np.linspace(0, 100, len(drop_rel))
            safe_drop = [max(float(v), 0.01) for v in drop_rel]
            ax_drop.plot(drop_axis, safe_drop, label=label_text, color=color, linestyle=ls, marker=marker, markersize=3.5, alpha=0.85)

        if res.get("is_moe", False):
            ent_curve = res.get("layer_routing_entropy_curve", [])
            ortho_curve = res.get("layer_ortho_defect_curve", [])
            if ent_curve:
                d_moe = np.linspace(0, 100, len(ent_curve))
                ax_moe.plot(d_moe, ent_curve, label=f"{m_key} H_route (%)", color=color, linestyle="-", marker="s", markersize=4, alpha=0.9)
            if ortho_curve:
                d_ortho = np.linspace(0, 100, len(ortho_curve))
                ortho_sim_pct = [(1.0 - float(v)) * 100.0 for v in ortho_curve]
                ax_moe.plot(d_ortho, ortho_sim_pct, label=f"{m_key} Ortho (1-D)%", color=color, linestyle=":", marker="^", markersize=3.5, alpha=0.75)

        srank_curve = res.get("layer_srank_curve", [])
        if srank_curve:
            depth_s = np.linspace(0, 100, len(srank_curve))
            ax_srank.plot(depth_s, srank_curve, label=label_text, color=color, linestyle=ls, marker=marker, markersize=3.5, alpha=0.85)

        bpb_val = res.get("val_bpb", 0.0)
        act_b = res.get("active_params_b", 1.0)
        pareto_points.append((m_key, res["type"], act_b, bpb_val, color, marker))

    if pareto_points:
        bpb_vals_all = [p[3] for p in pareto_points if p[3] > 0]
        normal_bpb = [v for v in bpb_vals_all if v < 1.0] or bpb_vals_all
        y_min_plot = max(0.42, min(normal_bpb) - 0.045)
        y_max_plot = max(normal_bpb) + 0.045

        curated_offsets = {
            "Qwen-2.5-0.5B": ((8, 12), "left", "bottom"),
            "Qwen-2.5-1.5B": ((-2, -26), "center", "top"),
            "Qwen-2.5-3B": ((16, 8), "left", "bottom"),
            "Qwen-2.5-7B": ((-6, 18), "right", "bottom"),
            "SmolLM2-1.7B": ((12, 24), "center", "bottom"),
            "Pythia-2.8B": ((14, 0), "left", "center"),
            "Granite-3.1-3B-A800M": ((-6, -24), "center", "top"),
            "OLMoE-1B-7B": ((-24, 18), "center", "bottom"),
            "Qwen1.5-MoE-A2.7B": ((2, 24), "center", "bottom"),
            "Moonlight-16B-A3B": ((14, -14), "left", "top"),
        }
        fallback_cycle = [
            ((10, 14), "left", "bottom"),
            ((-10, -22), "center", "top"),
            ((-16, 18), "right", "bottom"),
            ((14, -14), "left", "top"),
        ]

        frontier_cands = []
        for idx_p, (m_key, m_type, act_b, bpb_val, color, marker) in enumerate(pareto_points):
            plot_y = min(bpb_val, y_max_plot - 0.008)
            if bpb_val > y_max_plot:
                y_max_plot = max(y_max_plot, plot_y + 0.03)
            ax_pareto.scatter(act_b, plot_y, color=color, marker=marker, s=130, edgecolors="black", zorder=5)
            offset_xy, ha_val, va_val = curated_offsets.get(m_key, fallback_cycle[idx_p % len(fallback_cycle)])
            ax_pareto.annotate(
                f"{m_key}\n({bpb_val:.3f} BPB)",
                xy=(act_b, plot_y),
                xytext=offset_xy,
                textcoords="offset points",
                ha=ha_val,
                va=va_val,
                fontsize=7.5,
                fontweight="bold",
                color=color,
                arrowprops=dict(arrowstyle="-", color=color, lw=0.85, alpha=0.65, shrinkA=0, shrinkB=5),
                zorder=6,
            )
            frontier_cands.append((act_b, plot_y))

        frontier_cands.sort(key=lambda x: x[0])
        frontier_x, frontier_y = [], []
        best_y = float("inf")
        for px, py in frontier_cands:
            if py < best_y:
                frontier_x.append(px)
                frontier_y.append(py)
                best_y = py
        if len(frontier_x) >= 2:
            ax_pareto.plot(frontier_x, frontier_y, linestyle="-.", color="black", alpha=0.5, linewidth=1.5, label="Empirical Pareto Frontier")
        ax_pareto.set_xlim(0.40, max(p[2] for p in pareto_points) * 1.28)
        ax_pareto.set_ylim(y_min_plot, y_max_plot)

    ax_gor.set_title("Panel 1 (GOR-v2): Residual Conflict Rate R_conflict(l)", fontsize=11, fontweight="bold")
    ax_gor.set_xlabel("Normalized Layer Depth (%)", fontsize=9.5)
    ax_gor.set_ylabel("Destructive Conflict Rate (%)", fontsize=9.5)
    ax_gor.grid(True, linestyle=":", alpha=0.6)
    ax_gor.legend(fontsize=7, loc="best")

    ax_erank.set_title("Panel 2 (GOR-v2): Token-Normalized Effective Rank Ratio", fontsize=11, fontweight="bold")
    ax_erank.set_xlabel("Normalized Layer Depth (%)", fontsize=9.5)
    ax_erank.set_ylabel("erank_norm(X_l) / min(T, d)", fontsize=9.5)
    ax_erank.grid(True, linestyle=":", alpha=0.6)
    ax_erank.legend(fontsize=7, loc="best")

    ax_drop.set_title("Panel 3 (GOR-v2): Log-Scale Relative Layer-Drop Sensitivity", fontsize=11, fontweight="bold")
    ax_drop.set_xlabel("Normalized Layer Depth (%)", fontsize=9.5)
    ax_drop.set_ylabel("Relative Loss Increase Delta L / L_base (%, Log)", fontsize=9.5)
    ax_drop.set_yscale("log")
    ax_drop.grid(True, which="both", linestyle=":", alpha=0.6)
    ax_drop.legend(fontsize=7, loc="best")

    ax_moe.set_title("Panel 4 (OS-MoE): Normalized Routing Entropy & Gate Orthogonality", fontsize=11, fontweight="bold")
    ax_moe.set_xlabel("Normalized MoE Layer Depth (%)", fontsize=9.5)
    ax_moe.set_ylabel("Normalized Score (%)", fontsize=9.5)
    ax_moe.grid(True, linestyle=":", alpha=0.6)
    handles_moe, labels_moe = ax_moe.get_legend_handles_labels()
    if handles_moe:
        ax_moe.legend(fontsize=7, loc="best")

    ax_srank.set_title("Panel 5 (GEO-Muon): Weight Matrix Stable Rank Ratio srank(W_l)", fontsize=11, fontweight="bold")
    ax_srank.set_xlabel("Normalized Layer Depth (%)", fontsize=9.5)
    ax_srank.set_ylabel("Stable Rank Ratio ||W||_F^2 / (||W||_2^2 min(m,n))", fontsize=9.5)
    ax_srank.grid(True, linestyle=":", alpha=0.6)
    ax_srank.legend(fontsize=7, loc="best")

    ax_pareto.set_title("Panel 6 (Pareto): Active Parameters vs. Bits-Per-Byte (BPB)", fontsize=11, fontweight="bold")
    ax_pareto.set_xlabel("Active Parameters per Token (Billions, Log-Scale)", fontsize=9.5)
    ax_pareto.set_ylabel("Calibration Bits-Per-Byte (BPB, Tokenizer-Agnostic)", fontsize=9.5)
    ax_pareto.set_xscale("log")
    ax_pareto.grid(True, which="both", linestyle=":", alpha=0.6)
    handles_p, _ = ax_pareto.get_legend_handles_labels()
    if handles_p:
        ax_pareto.legend(fontsize=8, loc="best")

    buf = BytesIO()
    plt.savefig(buf, format="png", bbox_inches="tight")
    plt.close(fig)
    plot_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

    table_rows = []
    for m_key, res in results_data.items():
        type_badge = (
            "<span style='background:rgb(219,234,254); color:rgb(30,64,175); padding:3px 8px; border-radius:12px; font-weight:bold; font-size:11px;'>Dense</span>"
            if res["type"] == "Dense"
            else "<span style='background:rgb(243,232,255); color:rgb(107,33,168); padding:3px 8px; border-radius:12px; font-weight:bold; font-size:11px;'>MoE</span>"
        )
        moe_vio_str = f"{res.get('max_vio_global', 0.0)} / {res.get('max_vio_batch', 0.0)}" if res.get("is_moe") else "—"
        moe_ent_str = f"{res.get('routing_entropy_pct', 0.0)}%" if res.get("is_moe") else "—"
        moe_ortho_str = f"{res.get('router_ortho_defect', 0.0)}" if res.get("is_moe") else "—"

        row = f"""
        <tr style='border-bottom:1px solid rgb(226,232,240);'>
            <td style='padding:9px; font-weight:bold; text-align:left;'>{m_key}</td>
            <td style='padding:9px;'>{type_badge}</td>
            <td style='padding:9px;'>{res['total_params_b']}B / <strong style='color:rgb(15,118,110);'>{res['active_params_b']}B</strong></td>
            <td style='padding:9px;'>{res['num_layers']}</td>
            <td style='padding:9px; font-weight:bold; color:rgb(29,78,216);'>{res.get('val_bpb', 0.0)}</td>
            <td style='padding:9px; font-weight:bold;'>{res['val_ppl']}</td>
            <td style='padding:9px;'>{res['throughput_tok_s']}</td>
            <td style='padding:9px;'>{res.get('avg_conflict_rate_pct', 0.0)}% / {res.get('avg_ortho_innovation_pct', 0.0)}%</td>
            <td style='padding:9px;'>{res.get('avg_erank_norm_ratio', 0.0)}</td>
            <td style='padding:9px;'>{res.get('avg_bi', 0.0)} / {res.get('avg_cka_renewal', 0.0)}</td>
            <td style='padding:9px;'>{res.get('deep_layer_drop_rel_pct', 0.0)}%</td>
            <td style='padding:9px;'>{res.get('avg_srank_ratio', 0.0)}</td>
            <td style='padding:9px;'>{moe_ent_str}</td>
            <td style='padding:9px;'>{moe_ortho_str}</td>
            <td style='padding:9px;'>{moe_vio_str}</td>
        </tr>
        """
        table_rows.append(row)

    table_rows_html = "".join(table_rows)

    html_template = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Frontier LLMs 6-Panel Diagnostic Benchmark Report (v2)</title>
    <style>
        body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; background-color: rgb(248,250,252); color: rgb(30,41,59); margin: 0; padding: 24px; line-height: 1.5; }}
        .container {{ max-width: 1480px; margin: 0 auto; background: rgb(255,255,255); padding: 32px; border-radius: 12px; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1); border: 1px solid rgb(226,232,240); }}
        h1 {{ color: rgb(15,23,42); margin-top: 0; font-size: 25px; border-bottom: 2px solid rgb(37,99,235); padding-bottom: 12px; }}
        h2 {{ color: rgb(30,58,138); font-size: 18px; margin-top: 28px; border-bottom: 1px solid rgb(203,213,225); padding-bottom: 6px; }}
        table {{ width: 100%; border-collapse: collapse; margin-top: 16px; font-size: 12px; text-align: center; }}
        th {{ background: rgb(241,245,249); color: rgb(51,65,85); font-weight: 600; padding: 10px 6px; border-bottom: 2px solid rgb(203,213,225); }}
        td {{ padding: 7px 6px; }}
        .figure-box {{ text-align: center; margin-top: 24px; background: rgb(255,255,255); padding: 16px; border-radius: 8px; border: 1px solid rgb(226,232,240); }}
        .figure-box img {{ max-width: 100%; height: auto; border-radius: 6px; }}
        .callout {{ background: rgb(240,253,244); border-left: 4px solid rgb(22,163,74); padding: 16px; margin: 20px 0; border-radius: 0 8px 8px 0; font-size: 13.5px; }}
        .callout-title {{ font-weight: bold; color: rgb(22,101,52); margin-bottom: 4px; }}
    </style>
</head>
<body>
<div class="container">
    <h1>🔬 Frontier LLMs 6-Panel Diagnostic Report v2 (0.5B – 16B: Iso-Tokenizer Dense & Sparse MoE)</h1>
    <p>Empirical ground-truth evaluation across 10 open-weight foundation models executed on NVIDIA A100 GPU in BF16 precision, profiling pre-norm residual dynamics, token-normalized effective rank, log-scale layer indispensability, MoE prototype orthogonality, spectral weight stable rank, and tokenizer-agnostic Bits-Per-Byte (BPB).</p>

    <div class="callout">
        <div class="callout-title">Methodological Rigor (v2 Diagnostic Upgrades)</div>
        All layer-wise hidden-state curves exclude final RMSNorm contamination via pre-norm forward hooks on <code>layers[-1]</code>. Effective Rank is computed on token-L2-normalized activations divided by <code>min(T, d)</code> to eliminate massive-activation sink bias. Cross-family efficiency is standardized via UTF-8 Bits-Per-Byte (BPB).
    </div>

    <h2>1. Comprehensive 6-Metric Cross-Model Data Matrix</h2>
    <table>
        <thead>
            <tr>
                <th style="text-align:left;">Model Name</th>
                <th>Class</th>
                <th>Total / Active</th>
                <th>Layers</th>
                <th>Val BPB</th>
                <th>Val PPL</th>
                <th>Tok/s</th>
                <th>Conflict / Ortho</th>
                <th>Avg erank_norm</th>
                <th>Avg BI / 1-CKA</th>
                <th>Deep Drop &Delta;L%</th>
                <th>Avg srank(W)</th>
                <th>MoE H_route</th>
                <th>MoE D_ortho</th>
                <th>MaxVio (G/B)</th>
            </tr>
        </thead>
        <tbody>
            {table_rows_html}
        </tbody>
    </table>

    <h2>2. Six-Panel (2x3) Mechanistic & Pareto Diagnostic Suite</h2>
    <div class="figure-box">
        <img src="data:image/png;base64,{plot_b64}" alt="Frontier Models 6-Panel Benchmark Suite v2">
    </div>
</div>
</body>
</html>
"""
    with open(output_html_path, "w", encoding="utf-8") as f:
        f.write(html_template)
    print(f"🎉 HTML Report successfully exported to: {output_html_path}")

    try:
        from google.colab import files
        files.download(output_html_path)
        print(f"📥 Colab download triggered for {output_html_path}")
    except Exception:
        pass

if benchmark_results:
    generate_html_benchmark_report(benchmark_results, "frontier_models_benchmark_report.html")
